In [1]:
# Filemame......: 44 Securing Infrastructure for AI Applications
# Language......: Python
# Tools.......... Visual Source Code (VSC) & Colab Google
# Class.........: ITAI 1372 Artificial Intelligence in Cybersecurity
# Semster.......: Fall 2026
# Class Type....: Online
# Instructor....: Hien Nguyen
# Student.......: Olis Bahari
# Version.......: V1.0

In [2]:
#from google.cloud import auth
#auth.authenticate_user()
import google.auth

credentials, project_id = google.auth.default()

print("Authentication successful!")
print("Project ID:", project_id)
print("Quota Project:", credentials.quota_project_id)

Authentication successful!
Project ID: dynamic-circlet-j41fc
Quota Project: dynamic-circlet-j41fc


In [3]:
# Google Cloud DLP configuration
LOCATION = "global"

In [4]:
# Google Cloud DLP template IDs

INSPECT_TEMPLATE_ID = "pii-finder"
UNSTRUCTURED_TEMPLATE_ID = "de-identify-unstructured"
CLOUDSDK_CORE_PROJECT="dynamic-circlet-j41fc"
TEST_STRING = (
    "My name is John Smith. "
    "My email is john.smith@example.com. "
    "My phone number is 415-555-1234."
)

print(TEST_STRING)

My name is John Smith. My email is john.smith@example.com. My phone number is 415-555-1234.


In [5]:
import os
import google.auth

credentials, project_id = google.auth.default()

PROJECT_ID = os.environ.get(
    "GOOGLE_CLOUD_PROJECT",
    project_id
)

print("Project ID:", PROJECT_ID)

Project ID: dynamic-circlet-j41fc


In [6]:
from google.cloud import dlp_v2

dlp = dlp_v2.DlpServiceClient()

parent = f"projects/{PROJECT_ID}/locations/global"

response = dlp.list_deidentify_templates(
    request={"parent": parent}
)

templates = list(response.deidentify_templates)

for template in templates:
    print("Template:", template.name)

if not templates:
    print("No de-identification templates found.")

C:\Users\Lenovo\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:224: FutureWarning: You are using a non-supported Python version (3.10.8). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\Lenovo\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:224: FutureWarning: You are using a non-supported Python version (3.10.8). Google will not post any further updates to google.cloud.dlp_v2 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.dlp_v2.
  warnings.warn(message, FutureWarning)


No de-identification templates found.


In [7]:
def parent(project):
    return f"projects/{project}/locations/global"

def inspect_template_name(project):
    return (
        f"{parent(project)}/inspectTemplates/"
        f"{INSPECT_TEMPLATE_ID}"
    )

def deid_template_name(project, template_id):
    return (
        f"{parent(project)}/deidentifyTemplates/"
        f"{template_id}"
    )

In [8]:
import google.auth
import os

print("Windows user:", os.environ.get("USERNAME"))

credentials, project_id = google.auth.default()

print("Authentication successful!")
print("Project ID:", project_id)
print("Credentials type:", type(credentials).__name__)

Windows user: Lenovo
Authentication successful!
Project ID: dynamic-circlet-j41fc
Credentials type: Credentials


In [9]:
from google.cloud import dlp_v2

dlp = dlp_v2.DlpServiceClient()

print("Google Cloud DLP client initialized successfully!")

Google Cloud DLP client initialized successfully!


In [10]:
!python dlp_pipeline.py all --project dynamic-circlet-j41fc

  File "c:\Dropbox\Artifical Intellegence\Houston City College (HCC)\04 Artificial Intelligence in Cybersecurity\Olis-Bahari-CS-ITAI-1372\Lab\41 Securing Data Used for AI Applications\dlp_pipeline.py", line 2
    !pip install --quiet google-cloud-dlp google-cloud-storage
    ^
SyntaxError: invalid syntax


In [11]:
import argparse
import os
import subprocess
import sys
import time

from google.cloud import dlp_v2
from google.cloud import storage

In [12]:
UNSTRUCTURED_TEMPLATE_ID = "deid-unstructured"
STRUCTURED_TEMPLATE_ID = "deid-structured"
IMAGE_TEMPLATE_ID = "deid-image"

In [13]:
%pip install google-cloud-dlp
from google.cloud import dlp_v2
print("OK:", dlp_v2.InfoType)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
OK: <class 'google.cloud.dlp_v2.types.storage.InfoType'>


In [14]:
import os

project_id = os.environ.get('GOOGLE_CLOUD_PROJECT')

if not project_id:
    print("❌ GOOGLE_CLOUD_PROJECT environment variable is NOT set.")
elif project_id == 'your-gcp-project-id':
    print("⚠️ GOOGLE_CLOUD_PROJECT is still set to the default placeholder: 'your-gcp-project-id'.")
    print("Please edit cell 8c459dc1 with your actual GCP Project ID and re-run it.")
else:
    print(f"✅ GOOGLE_CLOUD_PROJECT is successfully configured to: '{project_id}'")

❌ GOOGLE_CLOUD_PROJECT environment variable is NOT set.


In [15]:
%%writefile dlp_pipeline.py
# Install necessary Google Cloud SDK libraries for Colab environment
!pip install --quiet google-cloud-dlp google-cloud-storage

#!/usr/bin/env python3
"""
Securing Data Used for AI Applications — automated PII sanitization pipeline.

Builds the same resources as the Google Codelab, but in code, using the
Sensitive Data Protection (Cloud DLP) API:

  1. Inspect template          pii-finder                 (what counts as sensitive)
  2. De-identify templates     de-identify-unstructured   (text/logs -> [INFO_TYPE])
                               de-identify-structured     (CSV user_id/message -> masked with #)
                               de-identify-image          (redaction boxes over PII in images)
  3. Inspection job            pii-remover                (scans gs://input-<project>, writes
                                                           de-identified copies to gs://output-<project>)
  4. Verification              before/after comparison of input vs. output files

Usage (run in Cloud Shell after ./setup.sh):
  python3 dlp_pipeline.py templates     # create / replace the 4 templates
  python3 dlp_pipeline.py test          # quick API sanity test on a sample string
  python3 dlp_pipeline.py job           # create the pii-remover job
  python3 dlp_pipeline.py wait          # poll the job until DONE, print findings
  python3 dlp_pipeline.py verify        # print before/after for text + CSV, list images
  python3 dlp_pipeline.py all           # templates -> test -> job -> wait -> verify
  python3 dlp_pipeline.py cleanup       # delete job + templates (buckets: see cleanup.sh)

Project is taken from --project, $GOOGLE_CLOUD_PROJECT, or `gcloud config`.
"""

from __future__ import annotations

import argparse
import os
import subprocess
import sys
import time

from google.api_core import exceptions as gexc
from google.cloud import dlp_v2, storage

LOCATION = "global"

INSPECT_TEMPLATE_ID = "pii-finder"
UNSTRUCTURED_TEMPLATE_ID = "de-identify-unstructured"
STRUCTURED_TEMPLATE_ID = "de-identify-structured"
IMAGE_TEMPLATE_ID = "de-identify-image"
JOB_ID = "pii-remover"

# infoTypes from the codelab's inspect template.
INFO_TYPES = [
    "CREDIT_CARD_EXPIRATION_DATE",
    "CREDIT_CARD_NUMBER",
    "DATE_OF_BIRTH",
    "DRIVERS_LICENSE_NUMBER",
    "EMAIL_ADDRESS",
    "GCP_API_KEY",
    "GCP_CREDENTIALS",
    "ORGANIZATION_NAME",
    "PASSWORD",
    "PERSON_NAME",
    "PHONE_NUMBER",
    "US_SOCIAL_SECURITY_NUMBER",
 ]

# CSV columns to sanitize. agent_id (employee address) is intentionally left alone.
STRUCTURED_FIELDS = ["user_id", "message"]

TEST_STRING = (
    "Hi, my name is Alex and my SSN is 555-11-5555. "
    "You can reach me at +1-555-555-5555."
)




Overwriting dlp_pipeline.py


In [16]:
# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def resolve_project(cli_value: str | None) -> str:
    project = cli_value or os.environ.get("GOOGLE_CLOUD_PROJECT")
    if not project:
        try:
            project = subprocess.check_output(
                ["gcloud", "config", "get-value", "project"],
                text=True, stderr=subprocess.DEVNULL,
            ).strip()
        except (OSError, subprocess.CalledProcessError):
            project = ""
    if not project:
        sys.exit("No project set. Use --project or `gcloud config set project <id>`.")
    return project


def parent(project: str) -> str:
    return f"projects/{project}/locations/{LOCATION}"


def inspect_template_name(project: str) -> str:
    return f"{parent(project)}/inspectTemplates/{INSPECT_TEMPLATE_ID}"


def deid_template_name(project: str, template_id: str) -> str:
    return f"{parent(project)}/deidentifyTemplates/{template_id}"


def info_types() -> list[dlp_v2.InfoType]:
    return [dlp_v2.InfoType(name=n) for n in INFO_TYPES]




In [17]:
# --------------------------------------------------------------------------- #
# Template definitions
# --------------------------------------------------------------------------- #
def build_inspect_config() -> dlp_v2.InspectConfig:
    return dlp_v2.InspectConfig(
        info_types=info_types(),
        min_likelihood=dlp_v2.Likelihood.POSSIBLE,
        include_quote=True,
    )


def build_unstructured_config() -> dlp_v2.DeidentifyConfig:
    """Free text / logs: replace each finding with its infoType name,
    e.g. 'John Doe' -> '[PERSON_NAME]'. Keeps sentence context for the model."""
    return dlp_v2.DeidentifyConfig(
        info_type_transformations=dlp_v2.InfoTypeTransformations(
            transformations=[
                dlp_v2.InfoTypeTransformations.InfoTypeTransformation(
                    primitive_transformation=dlp_v2.PrimitiveTransformation(
                        replace_with_info_type_config=dlp_v2.ReplaceWithInfoTypeConfig()
                    )
                )
            ]
        )
    )


def build_structured_config() -> dlp_v2.DeidentifyConfig:
    """CSV / tables: in user_id and message only, mask findings with '#',
    ignoring punctuation so the shape survives (user26@example.org ->
    ######@#######.###). Other columns (agent_id, dates, ids) keep their utility."""
    mask = dlp_v2.PrimitiveTransformation(
        character_mask_config=dlp_v2.CharacterMaskConfig(
            masking_character="#",
            characters_to_ignore=[
                dlp_v2.CharsToIgnore(
                    common_characters_to_ignore=dlp_v2.CharsToIgnore.CommonCharsToIgnore.PUNCTUATION
                )
            ],
        )
    )
    return dlp_v2.DeidentifyConfig(
        record_transformations=dlp_v2.RecordTransformations(
            field_transformations=[
                dlp_v2.FieldTransformation(
                    fields=[dlp_v2.FieldId(name=f) for f in STRUCTURED_FIELDS],
                    info_type_transformations=dlp_v2.InfoTypeTransformations(
                        transformations=[
                            dlp_v2.InfoTypeTransformations.InfoTypeTransformation(
                                primitive_transformation=mask
                            )
                        ]
                    ),
                )
            ]
        )
    )


def build_image_config() -> dlp_v2.DeidentifyConfig:
    """Images: draw black boxes over every detected infoType from pii-finder."""
    return dlp_v2.DeidentifyConfig(
        image_transformations=dlp_v2.ImageTransformations(
            transforms=[
                dlp_v2.ImageTransformations.ImageTransformation(
                    all_info_types=dlp_v2.ImageTransformations.ImageTransformation.AllInfoTypes(),
                    redaction_color=dlp_v2.Color(red=0.0, green=0.0, blue=0.0),
                )
            ]
        )
    )


DEID_TEMPLATES = {
    UNSTRUCTURED_TEMPLATE_ID: ("De-identify unstructured text", build_unstructured_config),
    STRUCTURED_TEMPLATE_ID: ("De-identify structured CSV", build_structured_config),
    IMAGE_TEMPLATE_ID: ("De-identify images", build_image_config),
}




In [18]:
# --------------------------------------------------------------------------- #
# Commands
# --------------------------------------------------------------------------- #
def cmd_templates(dlp: dlp_v2.DlpServiceClient, project: str) -> None:
    """Create all templates; if one already exists, replace it so reruns are idempotent."""
    name = inspect_template_name(project)
    tmpl = dlp_v2.InspectTemplate(display_name="PII finder", inspect_config=build_inspect_config())
    try:
        dlp.create_inspect_template(
            parent=parent(project), inspect_template=tmpl, template_id=INSPECT_TEMPLATE_ID
        )
        print(f"created  {name}")
    except gexc.AlreadyExists:
        dlp.update_inspect_template(name=name, inspect_template=tmpl)
        print(f"updated  {name}")

    for template_id, (display, builder) in DEID_TEMPLATES.items():
        name = deid_template_name(project, template_id)
        tmpl = dlp_v2.DeidentifyTemplate(display_name=display, deidentify_config=builder())
        try:
            dlp.create_deidentify_template(
                parent=parent(project), deidentify_template=tmpl, template_id=template_id
            )
            print(f"created  {name}")
        except gexc.AlreadyExists:
            dlp.update_deidentify_template(name=name, deidentify_template=tmpl)
            print(f"updated  {name}")


def cmd_test(dlp: dlp_v2.DlpServiceClient, project: str) -> None:
    """Run the codelab's test sentence through pii-finder + de-identify-unstructured."""
    resp = dlp.deidentify_content(
        request={
            "parent": parent(project),
            "inspect_template_name": inspect_template_name(project),
            "deidentify_template_name": deid_template_name(project, UNSTRUCTURED_TEMPLATE_ID),
            "item": {"value": TEST_STRING},
        }
    )
    print("BEFORE:", TEST_STRING)
    print("AFTER: ", resp.item.value)

    # Same idea for a one-row table, to show the structured template.
    table = dlp_v2.Table(
        headers=[dlp_v2.FieldId(name=h) for h in ("agent_id", "user_id", "message")],
        rows=[dlp_v2.Table.Row(values=[
            dlp_v2.Value(string_value="employee26@testdlp.com"),
            dlp_v2.Value(string_value="user26@example.org"),
            dlp_v2.Value(string_value="My updated ssn is 555-11-5555"),
        ])],
    )
    resp = dlp.deidentify_content(
        request={
            "parent": parent(project),
            "inspect_template_name": inspect_template_name(project),
            "deidentify_template_name": deid_template_name(project, STRUCTURED_TEMPLATE_ID),
            "item": {"table": table},
        }
    )
    row = [v.string_value for v in resp.item.table.rows[0].values]
    print("TABLE AFTER (agent_id, user_id, message):", row)


def cmd_job(dlp: dlp_v2.DlpServiceClient, project: str) -> None:
    """Create the pii-remover inspection job with a 'Make a de-identified copy' action."""
    input_bucket = f"input-{project}"
    output_bucket = f"output-{project}"

    storage_config = dlp_v2.StorageConfig(
        cloud_storage_options=dlp_v2.CloudStorageOptions(
            file_set=dlp_v2.CloudStorageOptions.FileSet(url=f"gs://{input_bucket}/**"),
            # De-identified copies require full scans: no sampling.
            bytes_limit_per_file=0,
            files_limit_percent=100,
            file_types=[
                dlp_v2.FileType.TEXT_FILE,
                dlp_v2.FileType.CSV,
                dlp_v2.FileType.IMAGE,
            ],
        )
    )

    deidentify_action = dlp_v2.Action(
        deidentify=dlp_v2.Action.Deidentify(
            transformation_config=dlp_v2.TransformationConfig(
                deidentify_template=deid_template_name(project, UNSTRUCTURED_TEMPLATE_ID),
                structured_deidentify_template=deid_template_name(project, STRUCTURED_TEMPLATE_ID),
                image_redact_template=deid_template_name(project, IMAGE_TEMPLATE_ID),
            ),
            cloud_storage_output=f"gs://{output_bucket}",
            file_types_to_transform=[
                dlp_v2.FileType.TEXT_FILE,
                dlp_v2.FileType.CSV,
                dlp_v2.FileType.IMAGE,
            ],
        )
    )

    job_config = dlp_v2.InspectJobConfig(
        storage_config=storage_config,
        inspect_template_name=inspect_template_name(project),
        actions=[deidentify_action],
    )

    job_name = f"{parent(project)}/dlpJobs/i-{JOB_ID}"
    try:
        job = dlp.create_dlp_job(
            request={"parent": parent(project), "inspect_job": job_config, "job_id": JOB_ID}
        )
        print(f"created  {job.name}  state={job.state.name}")
    except gexc.AlreadyExists:
        print(f"job already exists: {job_name}\n"
              f"delete it first with `python3 dlp_pipeline.py cleanup --job-only` to rerun.")


def _find_job(dlp: dlp_v2.DlpServiceClient, project: str) -> dlp_v2.DlpJob:
    # Job IDs created via the API are stored as 'i-<job_id>'.
    for candidate in (f"i-{JOB_ID}", JOB_ID):
        try:
            return dlp.get_dlp_job(name=f"{parent(project)}/dlpJobs/{candidate}")
        except gexc.NotFound:
            continue
    sys.exit(f"Job {JOB_ID} not found. Run `python3 dlp_pipeline.py job` first.")


def cmd_wait(dlp: dlp_v2.DlpServiceClient, project: str, timeout: int = 1800) -> None:
    terminal = {dlp_v2.DlpJob.JobState.DONE, dlp_v2.DlpJob.JobState.FAILED,
                dlp_v2.DlpJob.JobState.CANCELED}
    start = time.time()
    while True:
        job = _find_job(dlp, project)
        print(f"[{int(time.time() - start):>4}s] {job.name}  state={job.state.name}")
        if job.state in terminal:
            break
        if time.time() - start > timeout:
            sys.exit("Timed out waiting for job.")
        time.sleep(15)

    if job.state != dlp_v2.DlpJob.JobState.DONE:
        for err in job.errors:
            print("ERROR:", err.details.message)
        sys.exit(1)

    details = job.inspect_details.result
    print(f"\nBytes processed: {details.processed_bytes}")
    print("Findings by infoType:")
    for stat in sorted(details.info_type_stats, key=lambda s: -s.count):
        print(f"  {stat.info_type.name:<32} {stat.count}")


def _read_blob(client: storage.Client, bucket: str, path: str) -> str | None:
    blob = client.bucket(bucket).blob(path)
    if not blob.exists():
        return None
    return blob.download_as_text(errors="replace")


def cmd_verify(project: str, samples: int = 1) -> None:
    """Print before/after for text and CSV files; list redacted images to open in the console."""
    gcs = storage.Client(project=project)
    inp, out = f"input-{project}", f"output-{project}"

    out_paths = sorted(b.name for b in gcs.list_blobs(out))
    if not out_paths:
        sys.exit(f"gs://{out} is empty — has the job finished?")

    def show(prefix: str, n: int) -> None:
        for path in [p for p in out_paths if p.startswith(prefix)][:n]:
            before = _read_blob(gcs, inp, path)
            after = _read_blob(gcs, out, path)
            print("=" * 78)
            print(f"{path}\n--- BEFORE (gs://{inp}) ---\n{(before or '<missing>')[:800]}")
            print(f"--- AFTER  (gs://{out}) ---\n{(after or '<missing>')[:800]}")

    show("logs/", samples)
    show("csv/", samples)

    images = [p for p in out_paths if p.startswith("images/")]
    print("=" * 78)
    print(f"{len(images)} redacted images in gs://{out}/images/ — open in the console to view:")
    for p in images:
        print(f"  https://storage.cloud.google.com/{out}/{p}")

    in_paths = {b.name for b in gcs.list_blobs(inp)}
    missing = sorted(in_paths - set(out_paths))
    print(f"\n{len(out_paths)}/{len(in_paths)} input files have a de-identified copy.")
    for p in missing:
        print(f"  no output for: {p}")


def cmd_cleanup(dlp: dlp_v2.DlpServiceClient, project: str, job_only: bool) -> None:
    for candidate in (f"i-{JOB_ID}", JOB_ID):
        try:
            dlp.delete_dlp_job(name=f"{parent(project)}/dlpJobs/{candidate}")
            print(f"deleted job {candidate}")
        except gexc.NotFound:
            pass
    if job_only:
        return
    for template_id in DEID_TEMPLATES:
        try:
            dlp.delete_deidentify_template(name=deid_template_name(project, template_id))
            print(f"deleted {template_id}")
        except gexc.NotFound:
            pass
    try:
        dlp.delete_inspect_template(name=inspect_template_name(project))
        print(f"deleted {INSPECT_TEMPLATE_ID}")
    except gexc.NotFound:
            pass
    print("Buckets are not deleted here; run ./cleanup.sh to remove them.")




In [19]:
# --------------------------------------------------------------------------- #
def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("command", choices=["templates", "test", "job", "wait", "verify", "all", "cleanup"])
    ap.add_argument("--project", help="GCP project ID (default: gcloud config)")
    ap.add_argument("--samples", type=int, default=1, help="files per type to show in verify")
    ap.add_argument("--job-only", action="store_true", help="cleanup: delete only the job")

    # Check if running in a Jupyter/Colab kernel
    if len(sys.argv) > 0 and "kernel" in sys.argv[0]:
        # Force a default command to run within the notebook environment
        print("[Colab Notebook Mode]: Defaulting to 'test' command. Modify this array to change commands.")
        args = ap.parse_args(["test"])
    else:
        args = ap.parse_args()

    try:
        project = resolve_project(args.project)
    except SystemExit as e:
        print("\n[ERROR] No Google Cloud Project set!")
        print("Please configure your Google Cloud Project ID by running the following in a new cell:")
        print("    import os")
        print("    os.environ['GOOGLE_CLOUD_PROJECT'] = 'your-gcp-project-id'")
        return

    print(f"project: {project}\n")

    if args.command == "verify":
        cmd_verify(project, args.samples)
        return

    dlp = dlp_v2.DlpServiceClient()
    if args.command == "templates":
        cmd_templates(dlp, project)
    elif args.command == "test":
        cmd_test(dlp, project)
    elif args.command == "job":
        cmd_job(dlp, project)
    elif args.command == "wait":
        cmd_wait(dlp, project)
    elif args.command == "cleanup":
        cmd_cleanup(dlp, project, args.job_only)
    elif args.command == "all":
        cmd_templates(dlp, project); print()
        cmd_test(dlp, project); print()
        cmd_job(dlp, project); print()
        cmd_wait(dlp, project); print()
        cmd_verify(project, args.samples)


if __name__ == "__main__":
    main()

[Colab Notebook Mode]: Defaulting to 'test' command. Modify this array to change commands.

[ERROR] No Google Cloud Project set!
Please configure your Google Cloud Project ID by running the following in a new cell:
    import os
    os.environ['GOOGLE_CLOUD_PROJECT'] = 'your-gcp-project-id'


In [20]:
# 1. Programmatically overwrite the global variables in our Colab notebook memory
import sys
import os

active_project = "dynamic-circlet-j41fc"
for name in ['PROJECT_ID', 'project_id']:
    globals()[name] = active_project
    setattr(sys.modules[__name__], name, active_project)

# 2. Update environment variables to direct any API clients and CLI commands
os.environ["GOOGLE_CLOUD_PROJECT"] = active_project
os.environ["CLOUDSDK_CORE_PROJECT"] = active_project

# 3. Clean up the local config and set project with explicit silent override flags to prevent prompts
!gcloud config set core/disable_prompts True --quiet
!gcloud config set project dynamic-circlet-j41fc --quiet

# 4. Enable the Sensitive Data Protection (DLP) API cleanly on our active project
!gcloud services enable dlp.googleapis.com --project=dynamic-circlet-j41fc --quiet

Updated property [core/disable_prompts].
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [ ]:
# for Olis Bahari 
# gcloud auth login
#  gcloud config set account bahari.olis@gmail.com